# SU2 Rotation

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from cmath import exp
from collections import Counter
from math import cos, pi, sin

import torch
from IPython.display import Markdown, SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler.registers.phase.su2_rotation import su2_rotation_calls, su2_rotation_ir
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg
from flagquantum.simulation.unitary import get_unitary

## `SU2RotationGate`
Implements an arbitrary SU(2) rotation.

The rotation is represented by the matrix:

$$
    e^{i \alpha}
    \begin{pmatrix}
    e^{i(\lambda + \phi)} \cos(\theta) & e^{i\phi} \sin(\theta) \\
    e^{i\lambda} \sin(\theta) & - \cos(\theta)
    \end{pmatrix}
$$

#### Parameters
 - `theta`: rotation angle $\theta$ in the above matrix.
 - `phi`: phase angle $\phi$ in the above matrix.
 - `lambd`: phase angle $\lambda$ in the above matrix.
 - `global_shift`: phase angle $\alpha$, i.e. apply a global phase shift of $e^{i \alpha}$. 

#### References
 - [Generalized Quantum Signal Processing](https://arxiv.org/abs/2308.01501). Motlagh and Wiebe. (2023). Equation 7.


In [ ]:
signature = Signature.build(q=1)

### Example Instances

In [ ]:
su2_rotation_gate = (pi / 4, pi / 2, pi / 2, 0)

In [ ]:
hadamard = (pi / 4, 0, 0, 0)

In [ ]:
t_gate = (0, 3 * pi / 4, 0, -3 * pi / 4)

#### Graphical Signature

In [ ]:
examples = {"SU2 rotation": su2_rotation_gate, "Hadamard": hadamard, "T": t_gate}
display(SVG(field_signature_svg([(name, signature, {}) for name in examples])))

### Call Graph

In [ ]:
theta, phi, lambd, shift = su2_rotation_gate
calls = su2_rotation_calls(theta, phi, lambd, global_shift=shift)
root = "SU2 rotation"
graph = symbolic_call_graph(root, {root: Counter(calls)}, max_depth=1)
assert sum(graph.leaf_counts.values()) == 4
display(SVG(render_call_graph_svg(export_call_graph_dot(root, graph.direct_calls))))
display(Markdown(format_call_graph_markdown(root, graph.direct_calls, graph.leaf_counts)))

### Append and preserve absolute phase

Bind each example to a physical wire. Compare its complete complex matrix against the formula above, check Hadamard and T against the native gates, and append the exact adjoint. The resource graph counts one logical scalar and three rotations before gate classification. The scalar has zero modeled QEC cost and uses the existing phase carrier in the physical stream.

In [ ]:
dtype = torch.complex128
for name, (theta, phi, lambd, shift) in examples.items():
    gates = su2_rotation_ir(0, theta, phi, lambd, global_shift=shift)
    circuit = Circuit(1, dtype=dtype).append(FieldOperation(1, gates))
    expected = exp(1j * shift) * torch.tensor(
        [[exp(1j * (lambd + phi)) * cos(theta), exp(1j * phi) * sin(theta)],
         [exp(1j * lambd) * sin(theta), -cos(theta)]], dtype=dtype,
    )
    torch.testing.assert_close(get_unitary(circuit), expected, atol=2e-12, rtol=0)
    if name == "Hadamard":
        torch.testing.assert_close(expected, get_unitary(Circuit(1, dtype=dtype).h(0)), atol=2e-12, rtol=0)
    if name == "T":
        torch.testing.assert_close(expected, get_unitary(Circuit(1, dtype=dtype).t(0)), atol=2e-12, rtol=0)
    inverse = su2_rotation_ir(0, theta, phi, lambd, global_shift=shift, adjoint=True)
    circuit.append(FieldOperation(1, inverse))
    torch.testing.assert_close(get_unitary(circuit), torch.eye(2, dtype=dtype), atol=2e-12, rtol=0)
    print(name, "emitted", dict(Counter(g.name for g in gates)))
    print(name, "source call costs", [query_qec_gate_cost(call) for call in su2_rotation_calls(theta, phi, lambd, global_shift=shift)])